# 🍽️ Restaurant Finder — Multi-Agent System

نفس فكرة `AI_Agent_101` (CrewAI + Tavily + ScrapeGraph + Pydantic + JSON Outputs + Sequential Process)
لكن بدل مقارنة منتجات، بنبحث عن مطاعم ونحللها ونطلع تقرير HTML بأفضل الترشيحات.

| Agent | المسؤولية | الأداة | الـ Output |
|---|---|---|---|
| **A** — Search Query Agent | يقترح كلمات بحث مناسبة للمطاعم | — | `step_1_suggested_search_queries.json` |
| **B** — Search Agent | يبحث عن المطاعم | Tavily | `step_2_search_results.json` |
| **C** — Restaurant Scraping Agent | يستخرج بيانات المطاعم من صفحات الويب | ScrapeGraph | `step_3_scraped_restaurants.json` |
| **D** — Restaurant Analyst Agent | يحلل ويقارن المطاعم ويحدد الأفضل | — | `step_4_restaurant_analysis.json` |
| **E** — Report Agent | ينشئ تقرير HTML بالنتائج والتوصيات | — | `step_5_restaurant_report.html` |

In [ ]:
!pip install -qU crewai crewai-tools agentops

In [ ]:
!pip install -qU tavily-python scrapegraph-py

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
import agentops
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from scrapegraph_py import Client

import os
import json

In [ ]:
# please register Agentops account then go to
# (https://app.agentops.ai/get-started)
# to get the api key

os.environ["OPENAI_API_KEY"] = userdata.get('openai-colab')
os.environ["AGENTOPS_API_KEY"] = userdata.get('agentops-colab')

agentops.init(
    api_key=userdata.get('agentops-colab'),
    skip_auto_end_session=True,
    default_tags=['crewai', 'restaurants']
)

In [ ]:
output_dir = "./restaurant-agent-output"
os.makedirs(output_dir, exist_ok=True)

basic_llm = LLM(model="gpt-4o", temperature=0)
search_client = TavilyClient(api_key=userdata.get('tvly-search'))
scrape_client = Client(api_key=userdata.get('scrapegraph'))

In [ ]:
no_keywords = 10

about_user = (
    "The user is looking for a great restaurant to eat at. "
    "They care about food quality, customer ratings, fair prices and the overall experience. "
    "Recommendations must be based only on real data found on the web, never invented."
)

restaurant_context = StringKnowledgeSource(
    content=about_user
)

## Setup Agents

### Agent: A — Search Query Agent
يقترح كلمات بحث متنوعة ومحددة توصل لصفحات مطاعم حقيقية (مش مقالات أو قوائم عامة).

In [ ]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title="Suggested search queries to be passed to the search engine",
                               min_length=1, max_length=no_keywords)

search_queries_agent = Agent(
    role="Restaurant Search Queries Agent",
    goal="\n".join([
        "To provide a list of suggested search queries to be passed to the search engine.",
        "The queries must be varied and looking for specific restaurants."
    ]),
    backstory="The agent is designed to help in looking for restaurants by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm=basic_llm,
    verbose=True,
)

search_queries_task = Task(
    description="\n".join([
        "The user is looking for the best {cuisine} restaurants in {city}, {country_name}.",
        "Budget level: {budget}. Occasion / preferences: {preferences}.",
        "The user prefers results from these websites when possible: {websites_list}",
        "The goal is to reach as many relevant restaurants as possible so they can be compared later in another stage.",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contain specific dishes, neighborhoods, restaurant types or features. Avoid general keywords.",
        "The search query must reach a page of a specific restaurant (menu, reviews, details), and not a blog or a generic listing page."
    ]),
    expected_output="A JSON object containing a list of suggested search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent=search_queries_agent
)

### Agent: B — Search Agent
ياخد الكلمات من Agent A ويبحث بيها باستخدام Tavily.

In [ ]:
class SingleSearchResult(BaseModel):
    title: str
    url: str = Field(..., title="the page url")
    content: str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SingleSearchResult]

@tool
def search_engine_tool(query: str):
    """Useful for search-based queries. Use this to find current information about any query related pages using a search engine"""
    return search_client.search(query, max_results=5)

search_agent = Agent(
    role="Restaurant Search Agent",
    goal="To search for restaurants based on the suggested search queries",
    backstory="The agent is designed to help in looking for restaurants by searching the web based on the suggested search queries.",
    llm=basic_llm,
    verbose=True,
    tools=[search_engine_tool]
)

search_task = Task(
    description="\n".join([
        "The task is to search for restaurants based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any suspicious links, blogs, news articles, or generic 'top 10' listing pages. Keep pages of a single restaurant when possible.",
        "Ignore any search results with confidence score less than ({score_th}).",
        "Remove duplicate urls.",
        "The search results will be used to extract and compare restaurant details from different websites.",
    ]),
    expected_output="A JSON object containing the search results.",
    output_json=AllSearchResults,
    output_file=os.path.join(output_dir, "step_2_search_results.json"),
    agent=search_agent
)

### Agent: C — Restaurant Scraping Agent
يفتح الصفحات اللي طلعت من البحث ويستخرج بيانات كل مطعم باستخدام ScrapeGraph.

In [ ]:
class ExtractedRestaurant(BaseModel):
    page_url: str = Field(..., title="The original url of the restaurant page")
    restaurant_name: str = Field(..., title="The name of the restaurant")
    cuisine_types: List[str] = Field(default_factory=list, title="Cuisine types, e.g. Italian, Seafood")
    address: Optional[str] = Field(default=None, title="Full address. Set to None if not found")
    phone: Optional[str] = Field(default=None, title="Phone number. Set to None if not found")
    rating: Optional[float] = Field(default=None, title="Average customer rating. Set to None if not found")
    reviews_count: Optional[int] = Field(default=None, title="Number of customer reviews. Set to None if not found")
    price_range: Optional[str] = Field(default=None, title="Price range or average price per person. Set to None if not found")
    opening_hours: Optional[str] = Field(default=None, title="Opening hours. Set to None if not found")
    popular_dishes: List[str] = Field(default_factory=list, title="Most popular or signature dishes", max_length=5)
    features: List[str] = Field(default_factory=list, title="Features like delivery, outdoor seating, family friendly, reservations", max_length=6)
    review_highlights: List[str] = Field(default_factory=list, title="Short highlights of what customers like or dislike", max_length=4)

class AllExtractedRestaurants(BaseModel):
    restaurants: List[ExtractedRestaurant]

@tool
def web_scraping_tool(page_url: str):
    """
    An AI Tool to help an agent to scrape a restaurant web page

    Example:
    web_scraping_tool(
        page_url="https://www.example.com/restaurants/some-restaurant"
    )
    """
    try:
        details = scrape_client.smartscraper(
            website_url=page_url,
            user_prompt="Extract ```json\n" + json.dumps(ExtractedRestaurant.model_json_schema()) + "```\n From the web page. If a value is not on the page, use null. Never guess."
        )
    except Exception as e:
        details = {"error": str(e)}

    return {
        "page_url": page_url,
        "details": details
    }

scraping_agent = Agent(
    role="Restaurant Scraping Agent",
    goal="To extract restaurant details from any website",
    backstory="The agent is designed to help in extracting required restaurant details (rating, price, address, popular dishes, ...) from any web page url. These details will be used to decide which restaurant is the best.",
    llm=basic_llm,
    tools=[web_scraping_tool],
    verbose=True,
)

scraping_task = Task(
    description="\n".join([
        "The task is to extract restaurant details from restaurant page urls.",
        "The task has to collect results from multiple page urls.",
        "Pick the best {top_restaurants_no} pages from the search results (single restaurant pages only) and scrape them.",
        "If a page fails or contains no restaurant data, skip it.",
        "Do not invent any value that is not on the page.",
    ]),
    expected_output="A JSON object containing restaurants details",
    output_json=AllExtractedRestaurants,
    output_file=os.path.join(output_dir, "step_3_scraped_restaurants.json"),
    agent=scraping_agent
)

### Agent: D — Restaurant Analyst Agent
يحلل ويقارن المطاعم ويرتبها ويحدد الأفضل.

In [ ]:
class RankedRestaurant(BaseModel):
    restaurant_name: str
    page_url: str
    rank: int = Field(..., title="Position in the final ranking. 1 is the best")
    overall_score: float = Field(..., title="Overall score from 0 to 10", ge=0, le=10)
    value_for_money_score: float = Field(..., title="Value for money score from 0 to 10", ge=0, le=10)
    pros: List[str] = Field(..., title="Strengths of this restaurant compared to the others")
    cons: List[str] = Field(..., title="Weaknesses or missing information")
    recommendation_notes: str = Field(..., title="One short paragraph on who should choose this restaurant")

class RestaurantAnalysis(BaseModel):
    ranked_restaurants: List[RankedRestaurant]
    best_overall: str = Field(..., title="Name of the best restaurant overall")
    best_value: str = Field(..., title="Name of the best value for money restaurant")
    analysis_summary: str = Field(..., title="Short summary of the comparison")
    key_observations: List[str] = Field(..., title="Important trends or observations across the restaurants")

restaurant_analyst_agent = Agent(
    role="Restaurant Analyst Agent",
    goal="To analyze and compare restaurants and decide which ones are the best for the user",
    backstory="The agent is a food critic and data analyst. It compares restaurants fairly using ratings, number of reviews, prices, popular dishes and features, and it is honest about missing data.",
    llm=basic_llm,
    verbose=True,
)

restaurant_analysis_task = Task(
    description="\n".join([
        "The task is to analyze and compare the extracted restaurants.",
        "The user wants {cuisine} food in {city}, {country_name}. Budget level: {budget}. Preferences: {preferences}.",
        "Compare restaurants using: rating, number of reviews (a high rating with very few reviews is less reliable), price range vs budget, popular dishes, features, and match with the user's preferences.",
        "Give each restaurant an overall score and a value for money score (0 to 10).",
        "Rank ALL the restaurants from the best to the worst, and select the best overall and the best value.",
        "If data is missing, say it in the cons and do not invent it.",
        "Write in {language} language.",
    ]),
    expected_output="A JSON object containing the ranked restaurants and the analysis summary.",
    output_json=RestaurantAnalysis,
    output_file=os.path.join(output_dir, "step_4_restaurant_analysis.json"),
    agent=restaurant_analyst_agent
)

### Agent: E — Report Agent
ينشئ تقرير HTML احترافي بالنتائج والتوصيات.

In [ ]:
report_agent = Agent(
    role="Restaurant Report Author Agent",
    goal="To generate a professional HTML page for the restaurants recommendation report",
    backstory="The agent is designed to assist in generating a professional HTML page for the restaurants report after looking into the extracted data and the analysis.",
    llm=basic_llm,
    verbose=True,
)

report_task = Task(
    description="\n".join([
        "The task is to generate a professional HTML page for the restaurants recommendation report.",
        "You have to use Bootstrap CSS framework (CDN) for a better UI.",
        "Write the report in {language} language. If the language is Arabic, use dir='rtl'.",
        "Use the provided context about the user to make a specialized report.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: The best overall and best value restaurants in a highlighted box.",
        "2. Search Criteria: City, cuisine, budget and preferences.",
        "3. Methodology: How the restaurants were searched, extracted and compared.",
        "4. Comparison Table: Name, rating, reviews count, price range, address, phone, and the score of each restaurant.",
        "5. Restaurant Cards: For each restaurant (in rank order) show pros, cons, popular dishes, features and a link to its page.",
        "6. Analysis: Key observations and trends.",
        "7. Final Recommendations: Which restaurant to choose for which situation.",
        "8. Notes: Any missing data or limitations.",
        "Return ONLY the raw HTML code, without markdown code fences and without any explanation.",
    ]),
    expected_output="A professional HTML page for the restaurants report.",
    output_file=os.path.join(output_dir, "step_5_restaurant_report.html"),
    context=[scraping_task, restaurant_analysis_task],
    agent=report_agent,
)

## Run the AI Crew

In [ ]:
restaurant_crew = Crew(
    agents=[
        search_queries_agent,
        search_agent,
        scraping_agent,
        restaurant_analyst_agent,
        report_agent,
    ],
    tasks=[
        search_queries_task,
        search_task,
        scraping_task,
        restaurant_analysis_task,
        report_task,
    ],
    process=Process.sequential,
    knowledge_sources=[restaurant_context]
)

In [ ]:
crew_results = restaurant_crew.kickoff(
    inputs={
        "cuisine": "Italian",
        "city": "Cairo",
        "country_name": "Egypt",
        "budget": "medium",
        "preferences": "family friendly, good for dinner",
        "websites_list": ["www.tripadvisor.com", "www.elmenus.com", "www.talabat.com/egypt"],
        "language": "English",
        "no_keywords": 10,
        "score_th": 0.10,
        "top_restaurants_no": 8
    }
)

## عرض التقرير

In [ ]:
import re
from IPython.display import HTML, display

report_path = os.path.join(output_dir, "step_5_restaurant_report.html")

with open(report_path, "r", encoding="utf-8") as f:
    html = f.read()

# remove markdown fences if the model added them
html = re.sub(r"^```(?:html)?\s*|\s*```$", "", html.strip())

with open(report_path, "w", encoding="utf-8") as f:
    f.write(html)

display(HTML(html))